# backward-func-lookup composite — cx15: BackwardFuncLookup with asymmetric div_back0 / div_back1

> Composite procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Exercises 2 atoms together: `backward-func-lookup`, `arg-position-back-functions`
> Running the final beacon reports progress against all 2 subtopics.

**Why composite drills.** Single-atom drills test atomic skills in isolation. Composite drills test the COMPOSITION — how atoms wire together in real ARENA code. Passing this drill demonstrates you can apply the atoms jointly, not just individually.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below. Beacon will report progress against ALL atoms exercised by this composite.

In [ ]:
# === Delta Drills auth (composite) ===
DD_TOKEN = ""  # paste token, then run
DD_PRIMARY_ATOM = "backward-func-lookup"
DD_ATOM_IDS = ["backward-func-lookup", "arg-position-back-functions"]
DD_SUBTOPICS = ["Backprop: BackwardFuncLookup", "Backprop: Arg-position back funcs"]
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Composing the registry with arg-position back fns

`BackwardFuncLookup` is the dispatcher — a dict keyed by `(forward_fn, arg_position) -> back_fn`. The arg-position part is non-trivial: asymmetric binary ops like `divide` have DIFFERENT back fns for arg-0 and arg-1:

- `div_back0(grad_out, out, x, y) = grad_out / y`             (d(x/y)/dx = 1/y)
- `div_back1(grad_out, out, x, y) = grad_out * (-x / y**2)`   (d(x/y)/dy = -x/y²)

Both register into the SAME lookup under the same `t.divide` key — distinguished only by argnum. A missing `(fn, argnum)` key raises `KeyError` with a diagnostic message so the user knows which registration they forgot.

### Composite Exercise — BackwardFuncLookup with asymmetric div_back0 / div_back1

**Atoms exercised together**: `backward-func-lookup`, `arg-position-back-functions`

Implement two pieces:

**1. `BackwardFuncLookup`** — a dict-backed registry with two methods:
  - `add_back_func(fwd, argnum, back_fn)` stores under key `(fwd, argnum)`.
  - `get_back_func(fwd, argnum)` returns the stored fn, raising `KeyError` with a message that mentions   both the forward fn AND the argnum on a miss.

**2. `div_back0` and `div_back1`** — the asymmetric per-arg back fns for `out = x / y` (no broadcasting; assume `x.shape == y.shape`). Then register BOTH into a `BackwardFuncLookup` under `t.divide` at argnum 0 and argnum 1.

The test exercises register + retrieve + miss + dispatcher-style call.

In [ ]:
# Fill in the function below, then run this cell. The test asserts the composition is correct.

class BackwardFuncLookup:
    def __init__(self):
        raise NotImplementedError

    def add_back_func(self, forward_fn, arg_position, back_fn):
        raise NotImplementedError

    def get_back_func(self, forward_fn, arg_position):
        raise NotImplementedError

def div_back0(grad_out, out, x, y):
    raise NotImplementedError

def div_back1(grad_out, out, x, y):
    raise NotImplementedError

def cx15_build_registry():
    """Return a BackwardFuncLookup with div_back0/div_back1 registered under t.divide."""
    raise NotImplementedError

def _test_cx15():
    BF = cx15_build_registry()
    assert isinstance(BF, BackwardFuncLookup)

    # (a) both argnums resolve to DIFFERENT fns
    f0 = BF.get_back_func(t.divide, 0)
    f1 = BF.get_back_func(t.divide, 1)
    assert f0 is not f1, 'asymmetric op: argnum 0 and 1 must map to different fns'

    # (b) the math is right
    x = t.tensor([6.0, 10.0]); y = t.tensor([2.0, 5.0])
    out = x / y
    g0 = f0(t.ones(2), out, x, y)
    g1 = f1(t.ones(2), out, x, y)
    assert t.allclose(g0, 1 / y), f'div_back0 wrong: {g0} vs {1/y}'
    assert t.allclose(g1, -x / y**2), f'div_back1 wrong: {g1} vs {-x/y**2}'

    # (c) miss raises KeyError with diagnostic mentioning fn AND argnum
    try:
        BF.get_back_func(t.sin, 0)
    except KeyError as e:
        msg = str(e)
        assert 'sin' in msg or 'torch' in msg, f'message missing fn: {msg!r}'
        assert '0' in msg, f'message missing argnum: {msg!r}'
    else:
        raise AssertionError('missing fn must raise KeyError')

    # (d) right fn / wrong argnum also raises
    try:
        BF.get_back_func(t.divide, 7)
    except KeyError:
        pass
    else:
        raise AssertionError('missing argnum must raise KeyError')

    # (e) dispatcher-style call: agree with torch.autograd
    x_ref = t.tensor([3.0, 8.0], requires_grad=True)
    y_ref = t.tensor([2.0, 4.0], requires_grad=True)
    z = (x_ref / y_ref).sum()
    z.backward()
    out_c = x_ref.detach() / y_ref.detach()
    g0_ours = BF.get_back_func(t.divide, 0)(t.ones(2), out_c, x_ref.detach(), y_ref.detach())
    g1_ours = BF.get_back_func(t.divide, 1)(t.ones(2), out_c, x_ref.detach(), y_ref.detach())
    assert t.allclose(g0_ours, x_ref.grad, atol=1e-6)
    assert t.allclose(g1_ours, y_ref.grad, atol=1e-6)

    # (f) two registries are independent
    BF2 = BackwardFuncLookup()
    try:
        BF2.get_back_func(t.divide, 0)
    except KeyError:
        pass
    else:
        raise AssertionError('separate instances must not share storage')
    _dd_passed.add('cx15')

_test_cx15()

<details><summary>Show solution — cx15</summary>

```python
class BackwardFuncLookup:
    def __init__(self):
        self.back_funcs = {}

    def add_back_func(self, forward_fn, arg_position, back_fn):
        self.back_funcs[(forward_fn, arg_position)] = back_fn

    def get_back_func(self, forward_fn, arg_position):
        key = (forward_fn, arg_position)
        if key not in self.back_funcs:
            raise KeyError(
                f'No back_fn for ({forward_fn!r}, argnum={arg_position}). '
                f'Did you forget add_back_func(fn, {arg_position}, ...)?'
            )
        return self.back_funcs[key]

def div_back0(grad_out, out, x, y):
    return grad_out / y

def div_back1(grad_out, out, x, y):
    return grad_out * (-x / (y * y))

def cx15_build_registry():
    bf = BackwardFuncLookup()
    bf.add_back_func(t.divide, 0, div_back0)
    bf.add_back_func(t.divide, 1, div_back1)
    return bf
```

The 2-tuple `(fn, argnum)` is the whole reason asymmetric ops register cleanly under one key prefix — nested dicts would force an extra `.get()` step and the diagnostic message gets noisier. Symmetric ops (add, multiply) still register both argnums even though the bodies are mirror images.
</details>

## Report completion

Run the cell below to send progress to Delta Drills. The beacon fires once and reports all 2 subtopics together.

In [ ]:
# === Delta Drills completion beacon (composite — fires for ALL atoms) ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'cx15'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'composite-drill:{DD_PRIMARY_ATOM}:cx15',
        'subtopics': ["Backprop: BackwardFuncLookup", "Backprop: Arg-position back funcs"],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported composite (atoms={DD_ATOM_IDS})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()